# FPViT with class weights (`--class-weight effective`)

**Question.** The shipped `baseline_paper` (FPViT, paper hyperparameters, no class weights) never predicts dermatofibroma (test recall 0). Does enabling the class weights already implemented in `train.py` improve the rare classes without losing accuracy?

**Design.** One change only with respect to `baseline_paper`: `--class-weight effective` (β = 0.999, the Training Agent's default). Same architecture, optimizer, learning rate, epochs, augmentation, checkpoint selection (`balanced_acc`) and early-stopping patience (15). Three seeds (42, 1, 2).

**Data.** Official DermaMNIST split, 28×28 (train 7007 / val 1003 / test 2005). Every choice (epoch, seed) is made on validation; the test split is used only to measure, once per run.

**Model file.** Only the selected model (seed 1) is shared; checkpoints are not in git (see `.gitignore`). Seeds 42 and 2 are documented by their results in `results/fpvit_class_weighted/`, and their checkpoints are available on request.

| Run | Drive folder | `best_model.pt` SHA-256 |
|---|---|---|
| seed 1 (selected on validation) | `fpvit_cw_effective_s1` | `<to fill>` |

Drive link: `<to fill>`


## 1. Class weights

Printed by `train.py` at the start of every run (`Loss weights (effective)`):

| Class | Train images | Weight | vs. nevi |
|---|---|---|---|
| akiec | 228 | 0.94 | 4.9× |
| bcc | 359 | 0.63 | 3.3× |
| bkl | 769 | 0.36 | 1.8× |
| df | 80 | 2.49 | 12.9× |
| mel | 779 | 0.35 | 1.8× |
| nv | 4693 | 0.19 | 1× |
| vasc | 99 | 2.03 | 10.5× |

Effective number of samples (Cui et al., CVPR 2019): weight ∝ (1 − β) / (1 − βⁿ), normalised to mean 1. Milder than inverse frequency, where a dermatofibroma would weigh 59× a nevus.

## 2. Reproduce

Run from the repository root, on a GPU (Colab T4: ~74 s per epoch, 45–100 min per run).

In [ ]:
# Training: one run per seed, everything else at the defaults of baseline_paper
for seed, out in [(42, "fpvit_cw_effective"), (1, "fpvit_cw_effective_s1"), (2, "fpvit_cw_effective_s2")]:
    !python train.py --class-weight effective --select-on balanced_acc --early-stop-patience 15 \
        --num-workers 2 --seed {seed} --out runs/{out} --log-file

In [ ]:
# Test: once per run, after training
for out in ["fpvit_cw_effective", "fpvit_cw_effective_s1", "fpvit_cw_effective_s2"]:
    !python evaluate_test.py --checkpoint runs/{out}/best_model.pt --num-workers 2

In [ ]:
# SHA-256 of the shared checkpoint, to fill the table above
!sha256sum runs/fpvit_cw_effective_s1/best_model.pt

## 3. Seed selection (validation only)

| Seed | Best epoch | Stopped at | Val balanced acc | Val acc | Val macro-F1 |
|---|---|---|---|---|---|
| 42 | 21 | 36 | 0.613 | 0.739 | 0.533 |
| **1** | **63** | **78** | **0.658** | **0.747** | **0.592** |
| 2 | 19 | 34 | 0.597 | 0.617 | 0.451 |
| `baseline_paper` | 34 | – | 0.533 | 0.737 | 0.486 |

Seed 1 has the highest validation balanced accuracy and is the selected model.

## 4. Test results (official split, 2005 images)

The table is recomputed from the saved reports by the next cell.

| Model | Accuracy | Balanced acc | Macro-F1 | Macro-AUC |
|---|---|---|---|---|
| `baseline_paper` (1 seed) | 0.721 | 0.536 | 0.478 | 0.919 |
| seed 42 | 0.728 | 0.602 | 0.512 | 0.915 |
| **seed 1 (selected on val)** | **0.733** | **0.617** | **0.557** | **0.931** |
| seed 2 | 0.595 | 0.579 | 0.422 | 0.902 |
| **Recipe, mean ± std (3 seeds)** | 0.685 ± 0.078 | **0.599 ± 0.019** | 0.497 ± 0.069 | 0.916 ± 0.014 |

In [ ]:
import json
from pathlib import Path
import numpy as np

RESULTS = Path("results/fpvit_class_weighted")   # one sub-folder per seed with test_report_dermamnist.json
runs = {"seed 42": "seed42", "seed 1": "seed1", "seed 2": "seed2"}
keys = [("test_acc", "acc"), ("test_balanced_acc", "bal_acc"), ("test_macro_f1", "macro_f1"), ("test_macro_auc", "auc")]

rows = {name: json.loads((RESULTS / folder / "test_report_dermamnist.json").read_text()) for name, folder in runs.items()}
print(f"{'':10}" + "".join(f"{short:>10}" for _, short in keys))
for name, r in rows.items():
    print(f"{name:10}" + "".join(f"{r[k]:10.3f}" for k, _ in keys))
print(f"{'mean':10}" + "".join(f"{np.mean([r[k] for r in rows.values()]):10.3f}" for k, _ in keys))
print(f"{'std':10}" + "".join(f"{np.std([r[k] for r in rows.values()], ddof=1):10.3f}" for k, _ in keys))

### Recall per class (test)

| Class | Test images | `baseline_paper` | seed 42 | **seed 1** | seed 2 | Mean |
|---|---|---|---|---|---|---|
| akiec | 66 | 0.42 | 0.62 | 0.47 | 0.62 | 0.57 ± 0.09 |
| bcc | 103 | 0.72 | 0.60 | 0.66 | 0.52 | 0.60 ± 0.07 |
| bkl | 220 | 0.45 | 0.52 | 0.44 | 0.42 | 0.46 ± 0.05 |
| df | 23 | 0.00 | 0.35 | 0.48 | 0.35 | 0.39 ± 0.08 |
| mel | 223 | 0.55 | 0.43 | 0.59 | 0.58 | 0.53 ± 0.09 |
| nv | 1341 | 0.82 | 0.83 | 0.83 | 0.63 | 0.76 ± 0.12 |
| vasc | 29 | 0.79 | 0.86 | 0.86 | 0.93 | 0.89 ± 0.04 |

Malignant lesions correctly classified (akiec + bcc + mel, 392 images): `baseline_paper` 224, seed 42 198, **seed 1 230**, seed 2 224.

## 5. Findings

1. **Class weights raise balanced accuracy by about 6 points, consistently across the three seeds** (0.599 ± 0.019 against 0.536), with no loss of AUC. Dermatofibroma goes from 0 to about 0.4 recall.
2. **The selected model (seed 1) beats `baseline_paper` on every aggregate test metric**, and recognises 230 malignant lesions against 224.
3. **The melanoma drop of seed 42 (0.43) is not confirmed** by seeds 1 and 2 (0.58–0.59).
4. **Seed 2 degenerates** (test accuracy 0.595, 272 nevi predicted as melanoma). Selecting the checkpoint on balanced accuracy alone rewards epochs that call everything a rare class, because balanced accuracy does not see false alarms. Macro-F1 does.
5. **Longer training helps.** Seed 1 ran to epoch 78 and is the best; seeds 42 and 2 stopped around epoch 35 with the learning rate still high.
6. **Validation is optimistic.** From validation to test, balanced accuracy drops by 1.1 (seed 42), 4.1 (seed 1) and 1.8 (seed 2) points: the winner of a selection is partly lucky. Only test numbers are reported as results.

## 6. Also tried: post-hoc logit adjustment (discarded)

Logits of `baseline_paper` shifted by −τ·log(train prior), τ chosen on validation (τ = 1.1). Test: balanced accuracy +1.9 (0.536 → 0.555) but accuracy −20.5 (0.721 → 0.516) and macro-F1 −11.2. Moving the decision threshold only moves the errors: the rare classes need retraining.

## 7. Limitations and next steps

- **Official split only.** These numbers are on the official DermaMNIST test, which has known cross-split leakage [2]. The model has not yet been scored on DermaMNIST-C val (the ensemble's common validation set) or on the DermaMNIST-E external test.
- **One seed for the baseline.** The comparison with `baseline_paper` is indicative.
- **Low precision on rare classes.** Many nevi are predicted as melanoma or dermatofibroma (false alarms).
- **Not in the ensemble.** The model enters the system only through the current promotion gate:
  `python -m train_agent.promotion <folder of fpvit_cw_effective_s1>`.
  The gate scores the case where all models vote (224 px images); on 28 px images only the 28 px models vote, where this model may matter most. That path is worth measuring separately.
- **Suggested for the next runs:** `--select-on macro_f1` and `--early-stop-patience 30`.